**Notebook version: 40** — Claude will state the version number after editing any cell in this notebook. If the number here doesn't match what Claude just said, your editor has a stale copy: close this tab and reopen it (don't rely on "close without saving" — autosave can already have overwritten the file with the stale version before you close) before running anything.

# fiftyone_review_processed.ipynb — browse CONVERTED (intermediate-schema) data

**When to use this:** *after* Stage 5.2 conversion. Loads a converted source's intermediate-schema output (DEC-046) directly — flat `images/`+`labels/`, canonical class ids, plain YOLO `.txt` labels, no train/val/test split yet. This is the stage Stage 5.3 (Box Audit) and Stage 5.5 (Model-Assisted Curation) actually work on, so this notebook is genuinely useful, not just a sanity check.

**What it can browse (set via `source_key`, cell below):**
- A Stage 5.2 processed source, e.g. `"exdark"`, `"roboflow_pothole_vhmow"` — the original single-source use case.
- `"merged"` — the post-cap, post-merge pool (`dataset/merged/`, Stage 5.6).
- `"final/train"` / `"final/val"` / `"final/test"` — the split output (Stage 5.8).

**Flagged-only mode (optional, `flagged_report_path`):** point it at a `box_audit.py`-style flagged-boxes report (e.g. `dataset/reports/elevator_status_s4lrk_flagged.json`) to load *only* the images with at least one flagged box, with the specific flagged detection(s) marked (`detection.flagged == True`) so they're distinguishable from an image's other, unflagged boxes — this is what Stage 5.3's "isolate real detection boxes from shape-defective ones" review actually needs. Only works against a plain processed-source `source_key` (flagged reports reference that source's own filenames, not `merged`/`final`'s source-prefixed ones).

**Why not FiftyOne's built-in YOLO importer:** `fo.types.YOLOv5Dataset` assumes a `dataset.yaml` + per-split (`train`/`val`/`test`) folder structure. That fits `final/<split>`, but not `dataset/processed/<source>/` or `dataset/merged/` (both deliberately flat, no split yet — DEC-036). Building the FiftyOne dataset directly from `images/`+`labels/` handles all three the same way, one code path instead of two.

**Not for:** raw acquisition-stage exports (`dataset/raw/<source>/`) — use `fiftyone_explore.ipynb` (COCO-style) or `fiftyone_preview.ipynb` (pre-pull) for those instead.

**Cell order, box-audit review section:** source config → build (guarded) → resume-without-rebuild (use this after a restart instead of re-running build) → persistent=True → backup snapshot (re-run anytime as a checkpoint) → launch App → show session → write-back → verify labels_reviewed/ visually. The mistakenness section below is separate end-to-end (its own build/App/write-back), see its own markdown header for how it differs.

In [1]:
# Imports
import json
import shutil
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    """Walk up from `start` to find the repo root (has config/ + AGENTS.md).

    Needed because notebooks live in notebooks/, not the repo root, and
    Jupyter's working directory depends on how it was launched -- this
    makes the scripts.* import below robust regardless of that.
    """
    for parent in [start, *start.parents]:
        if (parent / "config").is_dir() and (parent / "AGENTS.md").is_file():
            return parent
    raise RuntimeError("Could not locate repo root from notebook cwd.")


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

import fiftyone as fo

from scripts.utils.config_loader import get_canonical_names
from scripts.utils.file_utils import (
    build_stem_index, ensure_dir, final_dir, list_images, merged_dir, prefixed_filename, processed_dir, reports_dir,
)
from scripts.curate.run_mistakenness import (
    COCO_CROSSWALK, CANONICAL_KEY_TO_NAME, ELIGIBLE_CANONICAL_IDS, load_eligible_ground_truth,
)

CANONICAL_NAMES = get_canonical_names()

/opt/anaconda3/envs/second-vision/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Change this and re-run the cells below to browse a different pool.
# Matches dataset/processed/<source_key>/ by default -- e.g. "exdark",
# "dataset_ninja_pothole_detection", "open_images", "roboflow_pothole_vhmow" --
# or the literal strings "merged" (dataset/merged/) or "final/train" /
# "final/val" / "final/test" (dataset/final/<split>/).
#
# Keep this saved to match whatever you're actually reviewing right now --
# this is what a kernel restart + rerun-from-top falls back to. It sat on
# cv_project_hovyc long after review had moved on to other sources, which is
# part of why the App kept reverting to hovyc after restarts.
source_key = "roboflow_dlsu_d_vehicle_type_detection"

# Optional: path to a box_audit.py-style flagged-boxes report (list of
# {label_path, class, cx, cy, w, h, reasons}). Set to None for normal
# browsing of every image in source_key. Only valid when source_key is a
# plain processed source (not "merged"/"final/*").
flagged_report_path = None

# Only applies when flagged_report_path is None (flagged mode is already
# scoped to the merged pool for free -- box_audit.py --pool merged only
# ever scans dataset/merged/, DEC-075). Restricts the full-pool browse to
# images this source actually contributed to dataset/merged/ -- i.e. what
# cap_per_class.py selected, not everything ever acquired. Real example
# that's why this exists: door_detection_zqt59 has 4,493 processed images,
# but only 3,460 were selected into the Doors class's cap; the other 1,033
# have no path to reaching the trained model (DEC-078 doesn't backfill an
# excluded slot from unselected candidates), so reviewing them is wasted
# time -- same reasoning DEC-075 already applied to box_audit.py, just
# missed here until caught for real. Set False to see the full raw pool
# anyway (e.g. before deciding whether a source needs a manual cap bump).
restrict_to_merged = True

# Added 2026-08-21 (DEC-088) for the post-dedup review pass: hide images that
# dedup.py (Stage 5.6) flagged as a duplicate of another image already kept,
# so review time isn't spent carefully cleaning boxes on a redundant copy
# that's functionally interchangeable with its "kept" representative.
# Student's explicit requirement: HIDE, not tag -- a duplicate-flagged image
# is not added to `dataset` at all, not shown with a badge you have to notice
# and skip yourself.
#
# Only the "duplicates" side of each dataset/reports/dedup_report.json group
# is hidden; each group's "kept" representative still loads normally, so
# exactly one copy per duplicate cluster remains reviewable. Covers both
# exact_duplicates (full-pool coverage) and near_duplicates (Stage 5.6's
# 6,000-image sample only -- a near-duplicate outside that sample was never
# checked and won't be hidden here either, same scope limit split.py's own
# duplicate-aware grouping already has, DEC-062).
#
# Tolerates dedup_report.json covering MORE images than are currently merged
# (e.g. reviewing a smaller cap+slack slice of a larger deduped pool, see
# docs/RUNPOD_DEDUP_PLAN.md) -- only refuses to run if the report covers
# FEWER than the current pool, which means it's genuinely stale relative to
# what's on disk now. Re-run dedup.py against the larger pool first if that
# happens, or leave this False.
hide_duplicates = True

# Added for the "come back later for a wider cap" workflow (docs/RUNPOD_DEDUP_
# PLAN.md): skip images that already have a dataset/processed/<source>/
# labels_reviewed/<stem>.txt entry from an earlier write-back run -- i.e.
# don't re-show images you've already been through, even if this session's
# scope (restrict_to_merged) is wider than it was last time (e.g. resuming a
# 5,500-per-class review at the full 10,000). A labels_reviewed/ entry exists
# for every image a write-back run processed, whether or not anything
# actually changed on that image -- so its mere presence is already a
# reliable "already looked at this" marker, nothing new to track.
#
# Depends on the write-back cell below never clearing labels_reviewed/ before
# writing -- otherwise a later write-back would delete the very entries this
# depends on for images outside the current (narrower) scope. Leave False for
# a normal first-pass review; set True specifically when resuming.
skip_previously_reviewed = False

# Optional: overlay a COCO-pretrained model's predictions alongside
# ground_truth, as a visual aid for spotting boxes that are MISSING (not
# wrong ones -- box_audit.py / flagged_report_path above already cover
# shape/size problems in boxes that already exist). Only helps for the
# 7/16 classes with a COCO analog (Person, Vehicle, Motorcycle, Bicycle,
# Animals, Chairs, Tables -- see run_mistakenness.py's COCO_CROSSWALK);
# leaves predictions empty for the other 9 classes, same ceiling as the
# mistakenness section below. Runs over every image in source_key, not a
# capped slice -- benchmarked for real on this machine at ~32 img/s, so
# even the largest single source (escalator_stairs, 7,560 images) is
# under 4 minutes. The mistakenness section's MISTAKENNESS_TOP_N=1000 cap
# is about the student's own review-TIME budget (DEC-074), not inference
# cost, so it doesn't apply here -- this is meant to run against whatever
# source you're already committed to reviewing in full.
show_predictions = True

# Only applies when show_predictions is True. Predictions at/above this
# confidence get pre-tagged 'accept' automatically at build time -- the same
# tag you'd otherwise click on by hand, and the same tag the write-back cell
# already looks for to promote a prediction into ground_truth. This does NOT
# skip review or promote anything by itself: pre-tagged predictions still show
# up as normal accepted predictions in the App, and nothing is promoted until
# the write-back cell actually runs -- so you can still spot-check each one and
# un-tag (or delete) any that are wrong first. Only saves you clicking accept
# on every obviously-correct high-confidence box by hand. Set to None to
# disable and tag everything manually, like before this existed.
#
# Picking this wrong is NOT a rebuild: every COCO-mapped prediction is stored
# regardless of confidence, so this only decides which ones start out tagged.
# If it turns out too low (or too high) mid-review, use the "adjust the
# auto-accept threshold" cell further down -- it re-applies a new value against
# the live dataset without re-running inference and without touching a single
# exclusion or hand-drawn box.
AUTO_ACCEPT_CONFIDENCE = 0.6

# Only applies when show_predictions is True. yolov8n re-detects objects that
# ground_truth ALREADY has a box for -- pure noise on any source whose own
# classes overlap COCO. A prediction found redundant is TAGGED 'dup_gt' and has
# its 'accept' withdrawn; it is never deleted, so a case where ground_truth is
# the wrong box is still there to find. Full measurements in DEC-098.
SUPPRESS_DUP_PREDICTIONS = True

# Same-class match: how much the boxes must agree on WHERE the object is.
# The best-same-class-IoU distribution across all review datasets is bimodal --
# 30.8% in [0.0,0.1) (same class, different object: a real find) and 64.9% at
# >= 0.5 -- with a 2.1% valley between, so anywhere in that gap behaves alike.
GT_MATCH_IOU = 0.5

# Above this a prediction is not a duplicate of a ground_truth box, it IS one,
# copied there by an earlier write-back's promotion. Geometric rather than
# tag-based on purpose: the 'promoted' tag postdates DEC-093, and predictions
# promoted before it carry none. Independent re-detection does not land on an
# existing box to six decimal places.
SELF_MATCH_IOU = 0.999

# Cross-class equivalences: a class with no COCO analog that COCO boxes under a
# near-neighbour name. Alias pairs match on ANY overlap, not on IoU, because a
# composite object gets boxed by its PARTS -- yolov8n boxes the motorcycle half
# or the sidecar cabin of a tricycle, a subset that scores low IoU against the
# whole tricycle box. No threshold separates those fragments from real objects;
# whether they overlap does.
#
# Deliberately minimal: Tricycle is 3,134 of the 3,406 cross-class matches
# measured across every review dataset. Each entry added here is a class
# confusion that can no longer be seen, and aliases self-disable on any source
# that labels the parts separately (see the dup_gt cell).
GT_CLASS_ALIASES = {
    "Tricycle": {"Vehicle", "Motorcycle", "Bicycle"},
}


In [4]:
# Build the FiftyOne dataset directly from images/+labels/ -- no network
# calls (aside from the optional predictions overlay below).
#
# NOT safe to blindly re-run once a review is underway: this cell wipes and
# rebuilds `dataset_name` from scratch every time (fo.delete_dataset + fresh
# fo.Dataset), discarding every tag/edit made since the last build. That
# exact mistake has already destroyed real review work twice in this project
# (door_detection_zqt59's tags after a kernel restart, then
# cv_project_hovyc's 71 exclusions + 277 accepted predictions the same way)
# -- both times because this cell got re-run to "get `dataset` back" instead
# of re-binding to the existing one. The guard below refuses to do that
# silently; see FORCE_REBUILD just above the delete/create lines. If the
# guard stops you, use the "resume without rebuilding" cell right below this
# one instead of touching FORCE_REBUILD.
#
# Persistence note: this dataset is created with persistent=True (see the
# create line below), so its edits survive a kernel restart and the rebuild
# guard is armed immediately rather than after a manual step. Edits made in the
# App -- adding, moving, deleting a box via the "Annotate" tab, or tagging --
# auto-save to FiftyOne's backing DB as you make them, so they are real in
# `dataset` right away. They are NOT on disk until the write-back step below
# copies them into dataset/processed/<source>/labels_reviewed/.
#
# Three independent layers protect review work, and they guard different things:
#   1. persistent=True  -- survives a kernel restart.
#   2. the FORCE_REBUILD guard below -- refuses to delete a persistent dataset,
#      which is what stops a stray re-run of this cell from wiping the review.
#   3. the snapshot backup cell after the App launch -- the only thing that
#      helps if fo.delete_dataset() does run, whether by FORCE_REBUILD=True or
#      by hand. Run it before every write-back (DEC-092).
# Layer 3 has already been the sole reason a review survived; do not skip it.

# Resolve images_dir/labels_dir for the chosen pool. "merged" and "final/<split>"
# use file_utils' own path helpers (a different directory layout, source-prefixed
# filenames); everything else is treated as a Stage 5.2 processed source key.
if source_key == "merged":
    images_dir = merged_dir() / "images"
    labels_dir = merged_dir() / "labels"
elif source_key.startswith("final/"):
    split = source_key.split("/", 1)[1]
    images_dir = final_dir(split) / "images"
    labels_dir = final_dir(split) / "labels"
else:
    images_dir = processed_dir(source_key) / "images"
    labels_dir = processed_dir(source_key) / "labels"

# Optional flagged-only mode: restrict to images with >=1 flagged box, and mark
# which specific detection(s) triggered the flag -- and *why* (box_audit.py's
# reasons, e.g. "large_area_outlier (>0.31)" -- a near-full-image box, the
# classic classification-dataset-forced-into-detection symptom) -- so they
# stand out from an image's other, unflagged boxes and you're not guessing
# why something was flagged.
flagged_by_label_path: dict[str, list[dict]] = {}
if flagged_report_path is not None:
    if source_key == "merged" or source_key.startswith("final/"):
        raise ValueError(
            "flagged_report_path is only supported against a plain processed-source "
            "source_key -- merged/final use prefixed filenames a flagged report doesn't reference."
        )
    flagged_entries = json.loads(Path(flagged_report_path).read_text(encoding="utf-8"))
    for entry in flagged_entries:
        flagged_by_label_path.setdefault(entry["label_path"], []).append(entry)

# Restrict to images this source actually contributed to dataset/merged/ (see
# restrict_to_merged's comment above for why -- unselected candidates have no
# path to training and reviewing them is wasted time). Only applies to the
# full-pool, plain-processed-source case; flagged mode is already scoped this
# way for free, and merged/final are already the real pool by definition.
merged_filenames_for_source: set[str] | None = None
if (
    restrict_to_merged
    and flagged_report_path is None
    and source_key != "merged"
    and not source_key.startswith("final/")
):
    prefix = f"{source_key}__"
    merged_filenames_for_source = {
        p.name[len(prefix):] for p in (merged_dir() / "images").iterdir() if p.name.startswith(prefix)
    }

# hide_duplicates (DEC-088): resolve which of THIS source's images are the
# "duplicates" side of a dedup_report.json group, so the load loop below can
# skip adding them as samples entirely -- not tag, hide. Only meaningful
# against a plain processed-source source_key (dedup_report.json's filenames
# are already source-prefixed merged-pool names, same convention "merged"
# itself uses, so this doesn't apply there / isn't needed there).
duplicate_filenames_for_source: set[str] | None = None
if hide_duplicates:
    if source_key == "merged" or source_key.startswith("final/"):
        raise ValueError(
            "hide_duplicates is only meaningful against a plain processed-source source_key "
            "-- merged/final already show every image, duplicates included, by definition."
        )
    dedup_report_path = reports_dir() / "dedup_report.json"
    if not dedup_report_path.is_file():
        raise FileNotFoundError(
            f"{dedup_report_path} not found -- run scripts/preprocess/dedup.py (Stage 5.6) "
            f"first, or set hide_duplicates = False."
        )
    dedup_report = json.loads(dedup_report_path.read_text(encoding="utf-8"))
    current_merged_count = len(list_images(merged_dir() / "images", recursive=False))
    if current_merged_count > dedup_report["images_checked"]:
        raise RuntimeError(
            f"dedup_report.json reflects {dedup_report['images_checked']} images, but "
            f"dataset/merged/images/ currently has {current_merged_count} -- the report is "
            f"stale (the merged pool changed since dedup.py last ran, e.g. a cap_per_class.py/"
            f"merge.py rerun with new sources folded in). Re-run dedup.py against the current "
            f"pool before trusting hide_duplicates, or set it to False for this session."
        )
    if current_merged_count < dedup_report["images_checked"]:
        # Deliberate, supported case (RunPod union-coverage plan, docs/RUNPOD_DEDUP_PLAN.md):
        # dedup.py ran once against a larger cap+slack pool, and this session is reviewing a
        # smaller cap+slack slice of it -- the report legitimately covers more than what's
        # currently merged. Every filename lookup below is a set-membership check scoped to
        # THIS source's currently-loaded images, so entries for images outside the current
        # pool simply never match -- safe by construction, not just tolerated.
        print(
            f"  NOTE: dedup_report.json covers {dedup_report['images_checked']} images, a "
            f"SUPERSET of the current {current_merged_count}-image merged pool -- expected "
            f"when reviewing a smaller cap+slack slice of a larger deduped pool. Duplicate "
            f"hiding still applies correctly to whatever's actually in scope here."
        )
    # Near-duplicate false positives (DEC-090's threshold review, notebooks/
    # fiftyone_near_dup_inspection.ipynb) -- a "duplicate" member the student
    # already visually confirmed is NOT actually a duplicate must not be hidden
    # here, or hide_duplicates would silently re-hide 1,568+ real, distinct
    # images this review pass is specifically supposed to see. Only applies to
    # near_duplicates -- exact_duplicates are byte-identical, never went
    # through that review, and have no ambiguity to false-positive-check in
    # the first place.
    false_positive_path = reports_dir() / "near_duplicate_false_positives.json"
    near_dup_false_positives: set[str] = set()
    if false_positive_path.is_file():
        near_dup_false_positives = {
            entry["duplicate"]
            for entry in json.loads(false_positive_path.read_text(encoding="utf-8"))
        }

    # Every "duplicates" member from both checks -- the "kept" side of each group is
    # deliberately NOT collected here, so it still loads normally as this cluster's
    # one reviewable representative.
    duplicate_prefixed_names: set[str] = set()
    for group in dedup_report["exact_duplicates"]:
        duplicate_prefixed_names.update(group["duplicates"])
    for group in dedup_report["near_duplicates"]:
        duplicate_prefixed_names.update(
            d["filename"] for d in group["duplicates"]
            if d["filename"] not in near_dup_false_positives
        )
    prefix = f"{source_key}__"
    duplicate_filenames_for_source = {
        name[len(prefix):] for name in duplicate_prefixed_names if name.startswith(prefix)
    }

# skip_previously_reviewed: resolve which of THIS source's images already have a
# dataset/processed/<source>/labels_reviewed/<stem>.txt entry from an earlier
# write-back run, so the load loop below can skip re-adding them entirely -- not
# tag, skip, same "hide not tag" posture as hide_duplicates above. A
# labels_reviewed/ entry gets written for EVERY sample a write-back run
# processes, whether or not anything actually changed on that sample (see the
# write-back cell's unchanged counter) -- so file presence there is already a
# reliable "already looked at this" marker, nothing new to compute or track.
#
# Depends on the write-back cell below NOT clearing labels_reviewed/ before
# writing (fixed alongside this toggle) -- if it still wiped the folder first,
# the next write-back after a skip-filtered session would delete the very
# entries this depends on for images outside that session's (deliberately
# narrower) scope, silently losing the "already reviewed" record for them.
previously_reviewed_filenames_for_source: set[str] | None = None
if skip_previously_reviewed:
    if source_key == "merged" or source_key.startswith("final/"):
        raise ValueError(
            "skip_previously_reviewed is only meaningful against a plain processed-source "
            "source_key -- merged/final don't have a per-source labels_reviewed/ folder."
        )
    reviewed_labels_dir = processed_dir(source_key) / "labels_reviewed"
    previously_reviewed_filenames_for_source = (
        {p.stem for p in reviewed_labels_dir.glob("*.txt")} if reviewed_labels_dir.is_dir() else set()
    )

# Only set this to True when you actually want to discard whatever's
# currently in dataset_name (a persistent, previously-reviewed dataset
# included) and rebuild it blank from disk. Leave False for normal use --
# the guard below will tell you the safe alternative (fo.load_dataset)
# instead of silently wiping real work.
FORCE_REBUILD = False

dataset_name = f"review_{source_key.replace('/', '_')}"
if dataset_name in fo.list_datasets():
    existing = fo.load_dataset(dataset_name)
    if existing.persistent and not FORCE_REBUILD:
        raise RuntimeError(
            f"'{dataset_name}' already exists and is persistent -- real review work (tags, "
            f"accepted predictions, edited/added boxes) may be stored in it. Re-running this "
            f"cell would fo.delete_dataset() it and rebuild blank from disk, discarding all of "
            f"that. If you just want `dataset` bound again (e.g. after a kernel restart), run "
            f"this instead and skip the rest of this cell:\n"
            f"    dataset = fo.load_dataset({dataset_name!r})\n"
            f"If you genuinely want to discard it and start this source's review over from "
            f"disk, set FORCE_REBUILD = True above and re-run this cell."
        )
    fo.delete_dataset(dataset_name)
# persistent=True from the moment of creation, NOT flipped later by hand.
#
# The guard directly above only refuses a rebuild when `existing.persistent` is
# True. Creating the dataset as persistent=False meant a freshly built one was
# unprotected until the student remembered to run the `dataset.persistent = True`
# cell -- so any re-run of this cell in that window silently deleted the review.
# That is not hypothetical: it destroyed door_detection_zqt59's review on
# 2026-08-26 (12 exclusions, 9 Stairs / 3 Bicycle / 2 Pole hand-drawn boxes),
# recovered only because a snapshot backup happened to exist. It had already
# destroyed door_detection_zqt59's tags once before, and cv_project_hovyc's 71
# exclusions plus 277 accepted predictions, both the same way.
#
# Creating it persistent closes the window entirely: the guard is armed on the
# very next execution of this cell, with no manual step to forget. The tradeoff
# is that abandoned review datasets now survive kernel restarts and need
# fo.delete_dataset() to clear -- deliberately accepted, since an orphaned
# dataset costs disk while a wiped one costs hours of review.
dataset = fo.Dataset(dataset_name, persistent=True)

samples = []
image_paths_ordered = []
skipped_not_merged = 0
skipped_duplicate = 0
skipped_previously_reviewed = 0
for image_path in sorted(images_dir.iterdir()):
    if merged_filenames_for_source is not None and image_path.name not in merged_filenames_for_source:
        skipped_not_merged += 1
        continue  # not selected by cap_per_class.py -- won't reach training as things stand

    if duplicate_filenames_for_source is not None and image_path.name in duplicate_filenames_for_source:
        skipped_duplicate += 1
        continue  # dedup.py flagged this as a duplicate of another image already kept -- hidden, not shown

    if (
        previously_reviewed_filenames_for_source is not None
        and image_path.stem in previously_reviewed_filenames_for_source
    ):
        skipped_previously_reviewed += 1
        continue  # already has a labels_reviewed/ entry from an earlier session -- resuming, not re-reviewing

    label_path = labels_dir / f"{image_path.stem}.txt"
    label_filename = label_path.name

    if flagged_by_label_path and label_filename not in flagged_by_label_path:
        continue  # flagged mode: skip images with nothing flagged

    # Match by rounded (cx, cy, w, h), not raw float equality -- both sides come
    # from the same 6-decimal string formatting this project's converters use,
    # but comparing through a round-trip is more robust than trusting exact
    # float equality to hold.
    flagged_lookup = {
        (round(e["cx"], 6), round(e["cy"], 6), round(e["w"], 6), round(e["h"], 6)): e["reasons"]
        for e in flagged_by_label_path.get(label_filename, [])
    }

    sample = fo.Sample(filepath=str(image_path))
    # Stashed so a future write-back step knows exactly which label file a
    # sample's (possibly since-edited) detections came from, without having
    # to re-derive it from the image filename.
    sample["source_label_filename"] = label_filename
    detections = []
    if label_path.is_file():
        for line in label_path.read_text(encoding="utf-8").splitlines():
            parts = line.strip().split()
            if not parts:
                continue
            class_id = int(parts[0])
            cx, cy, w, h = (float(v) for v in parts[1:5])
            # Our labels are YOLO center-based (cx, cy, w, h); FiftyOne's
            # Detection.bounding_box is top-left-based (x, y, w, h) --
            # both normalized [0, 1], so just shift the origin.
            x, y = cx - w / 2, cy - h / 2
            reasons = flagged_lookup.get((round(cx, 6), round(cy, 6), round(w, 6), round(h, 6)))
            detections.append(
                fo.Detection(
                    label=CANONICAL_NAMES[class_id],
                    bounding_box=[x, y, w, h],
                    flagged=reasons is not None,
                    flag_reasons=", ".join(reasons) if reasons else "",
                )
            )

    sample["ground_truth"] = fo.Detections(detections=detections)
    samples.append(sample)
    image_paths_ordered.append(image_path)

# Optional predictions overlay (show_predictions, set above) -- a stock
# COCO-pretrained yolov8n's boxes for the 7/16 classes with a COCO analog
# (run_mistakenness.py's own COCO_CROSSWALK, imported rather than
# reimplemented so this can't drift from what that script actually does),
# as a visual aid for spotting boxes ground_truth is MISSING. Runs over
# every image in this source, not a capped slice -- see the note on
# show_predictions above for why that's fine cost-wise.
if show_predictions:
    from ultralytics import YOLO
    import torch

    model = YOLO("yolov8n.pt")
    coco_to_canonical = {c: CANONICAL_KEY_TO_NAME[key] for key, cs in COCO_CROSSWALK.items() for c in cs}
    device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

    paths = [str(p) for p in image_paths_ordered]
    print(f"Running yolov8n inference on {len(paths)} images for the predictions overlay (device={device})...")
    BATCH = 16
    predictions_by_index: dict[int, list[dict]] = {}
    auto_accepted_count = 0
    for i in range(0, len(paths), BATCH):
        batch = paths[i:i + BATCH]
        results = model.predict(batch, device=device, verbose=False)
        for offset, result in enumerate(results):
            dets = []
            names = result.names
            for box in result.boxes:
                coco_name = names[int(box.cls.item())]
                if coco_name not in coco_to_canonical:
                    continue
                x1, y1, x2, y2 = box.xyxyn[0].tolist()
                conf = float(box.conf.item())
                det = {
                    "label": coco_to_canonical[coco_name],
                    "bounding_box": [x1, y1, x2 - x1, y2 - y1],
                    "confidence": conf,
                }
                # Pre-tag obviously-correct high-confidence predictions 'accept' so
                # they don't all need clicking by hand -- still just a tag, still
                # fully reviewable/reversible in the App, still nothing promoted
                # until the write-back cell actually runs. See AUTO_ACCEPT_CONFIDENCE's
                # comment above for the full reasoning.
                if AUTO_ACCEPT_CONFIDENCE is not None and conf >= AUTO_ACCEPT_CONFIDENCE:
                    # 'auto_accept' is a marker, not a second decision: it records
                    # that THIS tag was applied by the threshold rather than clicked
                    # by hand. Nothing downstream reads it (every other check is a
                    # membership test for "accept"/"promoted", so it rides along
                    # harmlessly) -- it exists purely so the threshold-adjust cell
                    # below can raise the bar later without stripping accepts the
                    # student made deliberately on low-confidence boxes.
                    det["tags"] = ["accept", "auto_accept"]
                    auto_accepted_count += 1
                dets.append(det)
            predictions_by_index[i + offset] = dets

    for idx, sample in enumerate(samples):
        preds = predictions_by_index.get(idx, [])
        sample["predictions"] = fo.Detections(detections=[fo.Detection(**b) for b in preds])
    print(
        "Predictions overlay ready -- toggle the 'predictions' field visible in the App sidebar "
        "to compare against ground_truth. Tag a specific predicted box 'accept' (click it, tag "
        "from the Labels list) to promote it into ground_truth during write-back below."
    )
    if AUTO_ACCEPT_CONFIDENCE is not None:
        print(
            f"{auto_accepted_count} prediction(s) pre-tagged 'accept' automatically "
            f"(confidence >= {AUTO_ACCEPT_CONFIDENCE}) -- spot-check these in the App and "
            f"un-tag (or delete) any that are actually wrong before running write-back."
        )

dataset.add_samples(samples)

# Restore the `exclude` sample tag from this source's own excluded.json.
#
# Two things at once. (1) Prior exclusions become VISIBLE again after a rebuild
# -- write-back unions rather than replaces, so they were never lost from disk,
# but until now the App showed none of them and you could not tell what you had
# already excluded. (2) The reason this was added: it puts `exclude` into the
# dataset's tag vocabulary, which is what makes it a one-click checkbox in the
# App's tagging popup instead of a string retyped for every image. That is the
# same trick AUTO_ACCEPT_CONFIDENCE already uses for `accept` on predictions --
# pre-populate the tag from a known source of truth so the App offers it.
#
# FiftyOne offers a tag in that popup only when some sample actually carries it --
# verified against 1.20, not assumed: tag a sample, untag it, and distinct("tags")
# returns straight to []. There is no API to register an unused tag
# (`dataset.tags` is a DATASET-level label unrelated to sample tags, and
# ColorScheme.label_tags only colours tags that already exist). Hence the seed
# below for sources that have no exclusions yet.
restored_excluded = 0
if source_key != "merged" and not source_key.startswith("final/"):
    excluded_path = reports_dir() / f"{source_key.removeprefix('roboflow_')}_excluded.json"
    if excluded_path.is_file():
        previously_excluded = set(
            json.loads(excluded_path.read_text(encoding="utf-8"))["excluded_filenames"]
        )
        if previously_excluded:
            for sample in dataset:
                # excluded.json keys on the LABEL filename ("<stem>.txt"), not the image.
                if f"{Path(sample.filepath).stem}.txt" in previously_excluded:
                    sample.tags = [t for t in (sample.tags or []) if t != "exclude"] + ["exclude"]
                    sample.save()
                    restored_excluded += 1
            print(
                f"Restored 'exclude' on {restored_excluded} sample(s) from {excluded_path.name} "
                f"-- these were excluded in an earlier pass and stay excluded."
            )

# Seed `exclude` on the first sample when nothing was restored, so the tag exists
# and the App shows it as a checkbox instead of making you type it. Student's
# explicit call 2026-08-26, cost accepted knowingly.
#
# THE COST, stated plainly because it is real: this sample is genuinely tagged for
# exclusion. Leave it tagged and write-back will record it in excluded.json and
# merge.py will drop that image from the pool. Untag it in the App if it is an
# image worth keeping. Ceiling is one image per source, and only for sources whose
# excluded.json is still empty -- once a source has any real exclusion, the restore
# above covers the tag and this never fires again.
if restored_excluded == 0 and len(dataset) > 0:
    seed_sample = dataset.first()
    seed_sample.tags = [t for t in (seed_sample.tags or []) if t != "exclude"] + ["exclude"]
    seed_sample.save()
    print(
        f"Seeded 'exclude' on the first sample so the tag is clickable in the App:\n"
        f"    {Path(seed_sample.filepath).name}\n"
        f"    ^ UNTAG this in the App if you want to keep it -- left tagged, it gets excluded."
    )

# Declare the valid class list per label field -- without this, dataset.classes
# defaults to {} and the App's Annotate tab has nothing to populate a class
# dropdown with when you try to draw a brand-new box (existing-box edits and
# sample/label tagging work fine without it; only *creating* a new detection
# needs this). Must call dataset.save() after setting classes in-place, per
# FiftyOne's own docs.
dataset.classes = {
    "ground_truth": CANONICAL_NAMES,
    "predictions": CANONICAL_NAMES,
}
dataset.save()

# Keep an already-launched App session in sync with THIS dataset -- `session`
# and `dataset` are independent objects, so switching source_key and rebuilding
# does NOT automatically update what the App is showing. This is exactly what
# happened for real: after switching from cv_project_hovyc to stairs_i2yia,
# `dataset` correctly pointed at the new one but `session` kept showing hovyc
# until this line existed. No-op (silently skipped) the very first time this
# cell runs, before any session has been launched yet.
if "session" in globals():
    session.dataset = dataset
    print(f"App session re-synced to {dataset.name!r}.")

print(f"{len(dataset)} images loaded from {images_dir}")
if skipped_not_merged:
    print(
        f"({skipped_not_merged} images in dataset/processed/ skipped -- not selected by "
        f"cap_per_class.py into dataset/merged/, so not reviewed. Set restrict_to_merged=False "
        f"above to see them anyway.)"
    )
if skipped_duplicate:
    print(
        f"({skipped_duplicate} images hidden -- flagged as a duplicate by dedup.py, "
        f"the cluster's 'kept' representative is what's shown instead. Set hide_duplicates=False "
        f"above to see them anyway.)"
    )
if skipped_previously_reviewed:
    print(
        f"({skipped_previously_reviewed} images skipped -- already have a labels_reviewed/ entry "
        f"from an earlier write-back run. Set skip_previously_reviewed=False above to see them "
        f"anyway (e.g. to double-check earlier work).)"
    )
if flagged_by_label_path:
    total_flagged_boxes = sum(len(v) for v in flagged_by_label_path.values())
    print(
        f"Flagged-only mode: {len(flagged_by_label_path)} images, {total_flagged_boxes} flagged boxes "
        f"-- marked detections have flagged == True, with the reason(s) on flag_reasons "
        f"(visible in the App's sample modal, under the detection's attributes)"
    )

  NOTE: dedup_report.json covers 67109 images, a SUPERSET of the current 45132-image merged pool -- expected when reviewing a smaller cap+slack slice of a larger deduped pool. Duplicate hiding still applies correctly to whatever's actually in scope here.
Running yolov8n inference on 6357 images for the predictions overlay (device=mps)...
Predictions overlay ready -- toggle the 'predictions' field visible in the App sidebar to compare against ground_truth. Tag a specific predicted box 'accept' (click it, tag from the Labels list) to promote it into ground_truth during write-back below.
6506 prediction(s) pre-tagged 'accept' automatically (confidence >= 0.6) -- spot-check these in the App and un-tag (or delete) any that are actually wrong before running write-back.
 100% |███████████████| 6357/6357 [3.2s elapsed, 0s remaining, 1.7K samples/s]      
Seeded 'exclude' on the first sample so the tag is clickable in the App:
    02877d0f-5d09-4e52-ac00-d55ba9b1ab0d_JPG.rf.d1c6ba66ccbe3822d56f

**Suppress predictions that re-detect a box `ground_truth` already has.** Runs as part of the normal top-to-bottom flow; also safe to re-run on its own against a live dataset (e.g. to try a different `GT_MATCH_IOU`) — it recomputes from scratch every time rather than accumulating.

Tags, never deletes. A redundant prediction keeps its box and its confidence and gains a `dup_gt` label tag; it only loses `accept`, so write-back won't promote it. Hide them in the App by filtering the `dup_gt` label tag in the sidebar.


In [10]:
# ---- Tag predictions that merely re-detect an existing ground_truth box ----
if SUPPRESS_DUP_PREDICTIONS and show_predictions:
    dataset.reload()

    def _iou(a, b):
        """Intersection over union of two FiftyOne [x, y, w, h] boxes (relative coords)."""
        ax, ay, aw, ah = a
        bx, by, bw, bh = b
        ix1, iy1 = max(ax, bx), max(ay, by)
        ix2, iy2 = min(ax + aw, bx + bw), min(ay + ah, by + bh)
        inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
        union = aw * ah + bw * bh - inter
        return inter / union if union > 0 else 0.0

    # Aliases are only safe for a composite class whose PARTS this source never
    # labels, decided from the dataset's own ground_truth rather than a hand-kept
    # list so it cannot go stale or be wrong for a source nobody has measured.
    #
    # dlsu_d_vehicle_type_detection labels Motorcycle (426) and Vehicle (426)
    # alongside Tricycle, with 112 + 30 of those real boxes sitting inside a
    # Tricycle box -- aliasing there would suppress genuine detections, so it
    # switches itself off. roitrikee and augmented_tricycle label only Tricycle,
    # so nothing of that kind exists to lose and it switches on.
    # Read the SOURCE's own labels, not the live ground_truth.
    #
    # "Does this source label the parts separately?" is a fact about its raw
    # export and never changes; the live ground_truth answers a different
    # question that drifts as you review. Drawing Vehicle/Motorcycle boxes into
    # roitrikee -- exactly what reviewing it entails -- made a ground_truth-based
    # check conclude the parts WERE labelled and switch the alias rule off,
    # un-tagging 432 predictions mid-review and putting every tricycle fragment
    # back on screen. Reading labels/ is stable against your own progress.
    present = set()
    for label_file in labels_dir.glob("*.txt"):
        for line in label_file.read_text(encoding="utf-8").splitlines():
            parts = line.split()
            if len(parts) == 5:
                present.add(CANONICAL_NAMES[int(parts[0])])
    if not present:
        # merged/final pools, or anything without a labels/ dir alongside.
        for sample in dataset:
            if sample.get_field("ground_truth"):
                for gt in sample.ground_truth.detections:
                    present.add(gt.label)
    alias_ok = {
        composite: set(aliases)
        for composite, aliases in GT_CLASS_ALIASES.items()
        if composite in present and not (set(aliases) & present)
    }
    print(f"aliases active for: {sorted(alias_ok) or 'nothing (parts are labelled separately)'}")

    marked = cleared = unmarked = skipped = overridden = 0
    for sample in dataset:
        preds = sample.get_field("predictions")
        if preds is None:
            continue
        gts = sample.ground_truth.detections if sample.get_field("ground_truth") else []
        changed = False
        for det in preds.detections:
            tags = list(det.tags or [])

            # Skipped BEFORE the test, not after. Write-back copies a promoted
            # prediction into ground_truth, so it thereafter matches its own copy
            # at IoU 1.0 and the test would be asking whether a box duplicates
            # itself. The tag only exists for datasets reviewed after DEC-093, so
            # the geometric check backstops it: cv_project_hovyc's 277 promotions
            # carry 'accept' with no 'promoted' tag and would otherwise have read
            # as 69.0% redundant against a true 2.9%.
            if "promoted" in tags or any(
                gt.label == det.label and _iou(det.bounding_box, gt.bounding_box) >= SELF_MATCH_IOU
                for gt in gts
            ):
                skipped += 1
                continue

            redundant = False
            for gt in gts:
                overlap = _iou(det.bounding_box, gt.bounding_box)
                if gt.label == det.label:
                    # Same class: needs real agreement on WHERE the object is.
                    redundant = redundant or overlap >= GT_MATCH_IOU
                elif det.label in alias_ok.get(gt.label, ()):
                    # Alias: ANY overlap. A composite object gets boxed by its
                    # parts -- yolov8n boxes the motorcycle half or the sidecar
                    # cabin of a tricycle, a subset scoring low IoU against the
                    # whole. Measured on roitrikee, those fragments spread evenly
                    # across every overlap band with no valley, so any threshold
                    # both leaves overlaps behind and eats real objects. Whether
                    # they overlap is the clean question; how much is not.
                    redundant = redundant or overlap > 0

            newly_marked = False
            if redundant and "dup_gt" not in tags:
                tags.append("dup_gt")
                marked += 1
                newly_marked = True
            elif not redundant and "dup_gt" in tags:
                # Recomputed clean -- e.g. GT_MATCH_IOU was raised, or the
                # ground_truth box this matched against was deleted.
                tags = [t for t in tags if t != "dup_gt"]
                unmarked += 1

            # Withdraw 'accept' only at the MOMENT dup_gt is first applied.
            #
            # An 'accept' sitting on an ALREADY-dup_gt prediction cannot have come
            # from the build cell -- this pass strips accept whenever it tags one --
            # so it can only have been added by hand in the App afterwards. That is
            # a deliberate override of this rule and must survive re-running it.
            # Stripping on every run instead of on transition silently undid those:
            # roitrikee had 6 such overrides on 2026-08-27, all Motorcycle/Vehicle
            # boxes on tricycles the student judged worth promoting.
            # Withdraw 'accept' ONLY when this pass applied it itself -- newly
            # marked dup_gt AND carrying the 'auto_accept' receipt.
            #
            # An 'accept' with no 'auto_accept' is a human decision and is never
            # withdrawn, however redundant the box now looks. Two ways it arises,
            # both real on roitrikee 2026-08-27: accept added by hand to an
            # already-dup_gt prediction (6 cases, a deliberate override), and a
            # hand-accepted prediction that only became redundant later because a
            # ground_truth box was drawn over it (36 cases). Withdrawing on
            # redundancy alone destroyed all 42 on the next run.
            if newly_marked and "accept" in tags and "auto_accept" in tags:
                tags = [t for t in tags if t not in ("accept", "auto_accept")]
                cleared += 1
            elif "dup_gt" in tags and "accept" in tags:
                overridden += 1

            if tags != list(det.tags or []):
                det.tags = tags
                changed = True
        if changed:
            sample.save()

    total = sum(
        len(s["predictions"].detections)
        for s in dataset
        if s.get_field("predictions") is not None
    )
    tagged = dataset.count_label_tags().get("dup_gt", 0)
    print(f"dup_gt: {tagged} of {total} predictions" + (f" ({100 * tagged / total:.1f}%)" if total else ""))
    print(f"  newly tagged this run        : {marked}")
    print(f"  'accept' withdrawn from them : {cleared}")
    if overridden:
        print(f"  your manual overrides kept  : {overridden}"
              "  <- accept added by hand on a dup_gt box")
    if unmarked:
        print(f"  no longer redundant, untagged: {unmarked}")
    if skipped:
        print(f"  already in ground_truth      : {skipped}  <- promoted by an earlier write-back")

    # The App does not notice DB writes made from this kernel on its own.
    if "session" in globals():
        session.refresh()
        print("App refreshed.")
else:
    print("skipped (SUPPRESS_DUP_PREDICTIONS or show_predictions is False)")


aliases active for: nothing (parts are labelled separately)
dup_gt: 7208 of 7937 predictions (90.8%)
  newly tagged this run        : 1254
  'accept' withdrawn from them : 671
App refreshed.


In [7]:
dataset.persistent = True

In [8]:
# Launch the App. If an App server is already running (it survives kernel
# restarts -- it's a separate long-lived process, not tied to this kernel's
# lifetime), fo.launch_app() reconnects to that existing server rather than
# resetting it, and the server keeps showing whatever dataset it was last
# bound to until told otherwise. The explicit session.dataset = dataset right
# after is what actually guarantees the App shows THIS dataset regardless --
# confirmed for real: after a kernel restart, `dataset` was correctly rebuilt
# for a new source but `session` still came back bound to the previous one
# until this line was added.
session = fo.launch_app(dataset, auto=False)
session.dataset = dataset
print(f"App session bound to {session.dataset.name!r}.")

Session launched. Run `session.show()` to open the App in a cell output.
App session bound to 'review_roboflow_dlsu_d_vehicle_type_detection'.


In [71]:
session

Dataset:          review_roboflow_door_detection_zqt59
Media type:       image
Num samples:      607
Selected samples: 0
Selected labels:  0
Session URL:      http://localhost:5151/

`dup_gt` marks predictions; it does not hide them. Run this to actually take them off the canvas while you review — and re-run with `HIDE_DUP_GT = False` to bring them back.


In [11]:
# ---- Move dup_gt predictions into their own FIELD (not a view) ----
# A `filter_labels` view hides redundant boxes in the grid but NOT reliably in the
# single-image modal, and FiftyOne 1.20 exposes no setting to change that. Views
# also live in session state, which any reconnecting client can clobber. Both
# failure modes have the same fix: don't ask the App to skip labels, give it a
# field that doesn't contain them.
#
# predictions      -- what you review. Untagged boxes, plus anything carrying
#                     'accept' so write-back still promotes your overrides.
# predictions_dup  -- the redundant ones, parked. Toggle it on in the App sidebar
#                     if you ever want to look; it is never drawn by default.
#
# Fully reversible: HIDE_DUP_GT = False moves everything back. Nothing is deleted
# and no box is edited -- boxes only change which field holds them.
HIDE_DUP_GT = True

# Declare the field before use -- sample.get_field() RAISES on a field that is
# not in the dataset schema rather than returning None.
if "predictions_dup" not in dataset.get_field_schema():
    dataset.add_sample_field(
        "predictions_dup",
        fo.EmbeddedDocumentField,
        embedded_doc_type=fo.Detections,
    )

moved = returned = 0
for sample in dataset:
    preds = sample.get_field("predictions")
    parked = sample.get_field("predictions_dup")
    live = list(preds.detections) if preds is not None else []
    held = list(parked.detections) if parked is not None else []

    if HIDE_DUP_GT:
        keep = []
        for det in live:
            tags = det.tags or []
            # 'accept' always stays in `predictions`. Write-back promotes on that
            # tag alone, so parking an accepted box would silently drop it from
            # the write-back -- including a deliberate accept-on-dup_gt override.
            if "dup_gt" in tags and "accept" not in tags:
                held.append(det)
                moved += 1
            else:
                keep.append(det)
        live, held = keep, held
    else:
        for det in held:
            live.append(det)
            returned += 1
        held = []

    if moved or returned:
        sample["predictions"] = fo.Detections(detections=live)
        sample["predictions_dup"] = fo.Detections(detections=held)
        sample.save()

dataset.reload()
n_live = sum(len(s["predictions"].detections) for s in dataset if s.get_field("predictions") is not None)
n_held = sum(len(s["predictions_dup"].detections) for s in dataset if s.get_field("predictions_dup") is not None)

if HIDE_DUP_GT:
    print(f"parked {moved} redundant prediction(s) into 'predictions_dup' this run")
else:
    print(f"returned {returned} prediction(s) to 'predictions'")
print(f"  predictions     (drawn)  : {n_live}")
print(f"  predictions_dup (parked) : {n_held}")
print("\nThe modal now draws only `predictions`. No view needed -- nothing to lose on a refresh.")
print("Re-running dupgt_code only sees `predictions`; set HIDE_DUP_GT = False and re-run this first if you want it to reconsider the parked ones.")

if "session" in globals():
    session.view = None
    session.refresh()
    print("App refreshed (view cleared -- it is no longer doing any work).")


parked 1254 redundant prediction(s) into 'predictions_dup' this run
  predictions     (drawn)  : 6683
  predictions_dup (parked) : 7208

The modal now draws only `predictions`. No view needed -- nothing to lose on a refresh.
Re-running dupgt_code only sees `predictions`; set HIDE_DUP_GT = False and re-run this first if you want it to reconsider the parked ones.
App refreshed (view cleared -- it is no longer doing any work).


In [25]:
import re
from collections import defaultdict
from pathlib import Path

def base_of(name):
    # Roboflow export convention: <original>_jpg.rf.<hash>.jpg
    m = re.match(r"^(.*?)_jpg\.rf\.[0-9a-f]+\.(jpg|jpeg|png)$", name, re.I)
    return m.group(1) if m else name

by_base = defaultdict(list)
for s in dataset:
    by_base[base_of(Path(s.filepath).name)].append((Path(s.filepath).name, s.id))

# Deterministic: keep the alphabetically-first variant of each base image
keep_ids = [sorted(v)[0][1] for v in by_base.values()]

deaug_view = dataset.select(keep_ids)
session.view = deaug_view
print(f"{len(dataset)} files -> {len(by_base)} distinct base images; showing one variant each.")


2329 files -> 1915 distinct base images; showing one variant each.


In [17]:
# Run THIS instead of re-running the build cell above whenever `dataset` needs to
# be bound again without rebuilding (e.g. after a kernel restart, or if the build
# cell's guard just stopped you). Never touches fo.delete_dataset -- purely a
# lookup by name, so it's always safe no matter what state review is in. Derives
# the name from source_key (matching the build cell's own dataset_name logic)
# rather than a hardcoded string, so this still works if you change source_key
# for a different review pass later.
dataset = fo.load_dataset(f"review_{source_key.replace('/', '_')}")

# Same App-sync as the build cell -- `session` doesn't follow `dataset` on its
# own, so without this the App keeps showing whatever it was bound to before.
# No-op the first time, before any session has been launched yet.
synced = False
if "session" in globals():
    session.dataset = dataset
    synced = True

print(
    f"{len(dataset)} samples bound from {dataset.name!r} (persistent={dataset.persistent})"
    + (" -- App session synced" if synced else "")
)

4217 samples bound from 'review_roboflow_revised_pedestrian_obstacle' (persistent=True) -- App session synced


**Changed your mind about `AUTO_ACCEPT_CONFIDENCE` mid-review?** Run this instead of rebuilding.

The build cell stores **every** COCO-mapped prediction with its `confidence` intact — the threshold only decides which ones start life tagged `accept`. So changing your mind is a re-tagging problem, not a rebuild problem: no inference re-runs, and nothing you've done in the App (exclusions, hand-drawn boxes, boxes you deleted) is at risk. Rebuilding, by contrast, deletes the dataset and everything in it that hasn't been written back yet — and since v28 the build cell's guard will refuse to do that anyway.


In [ ]:
# ---- Re-apply a different auto-accept threshold against the LIVE dataset ----
# Safe to run mid-review, as many times as you like. Only ever adds or removes
# the 'accept' tag on `predictions`; never touches ground_truth, sample tags,
# or anything on disk.

NEW_AUTO_ACCEPT_CONFIDENCE = 0.75

# Off by default, and the asymmetry is deliberate. Raising the bar only ever
# withdraws a suggestion the threshold made on its own. LOWERING it re-tags
# predictions that are currently untagged -- and a prediction you looked at and
# un-tagged by hand because it was wrong is indistinguishable from one that was
# never tagged at all (un-tagging in the App just removes the string; it leaves
# no "rejected" marker behind) -- EXCEPT where the threshold tagged it, which
# leaves 'auto_accept' as exactly that marker and is skipped below.
# already rejected. Turn this on only if you have not been un-tagging by hand,
# or are willing to re-check the newly-tagged ones.
ALLOW_TAG_UP = False

# When True, an 'accept' WITHOUT the 'auto_accept' marker is left alone -- i.e.
# a box you accepted yourself stays accepted even if it falls below the new bar.
PRESERVE_MANUAL_ACCEPTS = True

# Restrict this run to specific classes, e.g. {"Person"}. None = every class.
#
# Scopes BOTH directions: with {"Person"} set, a Vehicle prediction is neither
# un-tagged nor tagged up, whatever its confidence. Useful when one class needs a
# different bar from the rest -- yolov8n is far more confident on Person than on
# Chairs, so a single threshold is not equally strict across classes. Run it once
# per class with a different NEW_AUTO_ACCEPT_CONFIDENCE and the results compose;
# each run only ever touches the classes named here.
ADJUST_ONLY_CLASSES = None

dataset.reload()

# Datasets built before the 'auto_accept' marker existed carry no way to tell a
# threshold-applied tag from a hand-clicked one. Detect that up front rather
# than silently doing the wrong thing in either direction.
has_accept = has_marker = False
for sample in dataset:
    if not sample.has_field("predictions") or sample["predictions"] is None:
        continue
    for det in sample["predictions"].detections:
        tags = det.tags or []
        has_accept = has_accept or "accept" in tags
        has_marker = has_marker or "auto_accept" in tags
    if has_marker:
        break

# No 'auto_accept' markers anywhere means this dataset predates them (or was
# built with AUTO_ACCEPT_CONFIDENCE = None, so nothing was ever machine-tagged).
# Either way every 'accept' on it is a human decision.
#
# That is NOT an error -- PRESERVE_MANUAL_ACCEPTS=True simply means "keep all of
# them", which is exactly what it should do. This used to raise SystemExit and
# abort, which blocked the tag-UP path entirely even though tagging up strips
# nothing: revised_pedestrian_obstacle hit it with 4 hand-accepts and 0 markers
# while the student was only trying to lower the threshold.
#
# The down-path is skipped and the run continues. Set PRESERVE_MANUAL_ACCEPTS =
# False only if you want those accepts treated as machine-applied and stripped.
if has_accept and not has_marker:
    if PRESERVE_MANUAL_ACCEPTS:
        print(
            "No 'auto_accept' markers on this dataset -- every existing 'accept' is "
            "treated as hand-made and will be kept. Nothing will be un-tagged; set "
            "PRESERVE_MANUAL_ACCEPTS = False if you want them withdrawn instead."
        )
    else:
        print(
            "WARNING: no 'auto_accept' markers, and PRESERVE_MANUAL_ACCEPTS = False -- "
            "every accept below the new threshold will be withdrawn, including any you "
            "made by hand."
        )

untagged = tagged = kept_manual = kept_promoted = kept_rejected = 0
for sample in dataset:
    if not sample.has_field("predictions") or sample["predictions"] is None:
        continue
    changed = False
    for det in sample["predictions"].detections:
        if ADJUST_ONLY_CLASSES and det.label not in ADJUST_ONLY_CLASSES:
            continue
        tags = list(det.tags or [])
        conf = det.confidence
        if conf is None:
            continue

        # Already promoted = the box is in ground_truth already. Removing 'accept'
        # here would not take it back out, so report it instead of pretending.
        if "promoted" in tags:
            if conf < NEW_AUTO_ACCEPT_CONFIDENCE:
                kept_promoted += 1
            continue

        if "accept" in tags and conf < NEW_AUTO_ACCEPT_CONFIDENCE:
            if PRESERVE_MANUAL_ACCEPTS and "auto_accept" not in tags:
                kept_manual += 1
                continue
            det.tags = [t for t in tags if t not in ("accept", "auto_accept")]
            untagged += 1
            changed = True
        # 'dup_gt' excluded from tag-up on purpose: the dup_gt pass withdrew
        # accept from these because ground_truth already has the box, and that
        # stays true no matter what the confidence threshold is.
        #
        # 'auto_accept' WITHOUT 'accept' is a rejection RECORD, not a gap. Code
        # only ever writes or removes that pair together -- the build cell adds
        # both, the dup_gt pass and the branch above strip both -- so the only
        # way to hold one without the other is that the threshold suggested this
        # box and you un-tagged 'accept' by hand in the App. Re-tagging it here
        # would silently undo that call, which is the exact failure ALLOW_TAG_UP
        # is otherwise warned about above.
        elif (ALLOW_TAG_UP and "accept" not in tags and "dup_gt" not in tags
              and conf >= NEW_AUTO_ACCEPT_CONFIDENCE):
            if "auto_accept" in tags:
                kept_rejected += 1
                continue
            det.tags = tags + ["accept", "auto_accept"]
            tagged += 1
            changed = True
    if changed:
        sample.save()

print(f"threshold -> {NEW_AUTO_ACCEPT_CONFIDENCE}"
      + (f"   [only {sorted(ADJUST_ONLY_CLASSES)}]" if ADJUST_ONLY_CLASSES else ""))
print(f"  un-tagged (now below the bar) : {untagged}")
print(f"  newly tagged (tag-up)         : {tagged}"
      + ("" if ALLOW_TAG_UP else "   [ALLOW_TAG_UP is False]"))
if kept_manual:
    print(f"  left alone, accepted by hand  : {kept_manual}")
if kept_rejected:
    print(f"  left alone, you rejected these: {kept_rejected}"
          "  <- carry 'auto_accept' with no 'accept'")
if kept_promoted:
    print(f"  ALREADY PROMOTED, not revoked : {kept_promoted}"
          "  <- these are in ground_truth; delete the box in the App to remove one")

# The App does not notice DB writes made from this kernel on its own.
if "session" in globals():
    session.refresh()
    print("App refreshed.")


In [73]:
# Snapshot backup -- an extra layer of redundancy on TOP of persistent=True, not a
# replacement for it. persistent=True protects against a kernel restart; it does
# NOT protect against fo.delete_dataset() being called on this dataset by mistake,
# which is exactly how real review work (door_detection_zqt59's tags, then
# cv_project_hovyc's 71 exclusions + 277 accepted predictions) got destroyed twice
# in this project -- see the build cell's comment. This is the same export used
# both times to recover: tags, edited/added boxes, .classes, and the App's Schema
# Manager config (label_schemas) all round-trip correctly (verified directly).
# Media isn't re-copied (export_media=False) -- the images are already safe on
# disk, only the annotation data needs backing up.
#
# Safe to re-run anytime during a long review session as a checkpoint -- each run
# makes a new timestamped folder under dataset/backups/, nothing is ever
# overwritten or deleted.
import datetime

backup_dir = REPO_ROOT / "dataset" / "backups" / f"{dataset.name}_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
dataset.export(
    export_dir=str(backup_dir),
    dataset_type=fo.types.FiftyOneDataset,
    export_media=False,
)
print(f"Backed up {len(dataset)} samples (tags, boxes, schema) to {backup_dir}")

Exporting samples...
 100% |████████████████████| 607/607 [209.3ms elapsed, 0s remaining, 2.9K docs/s]       
Backed up 607 samples (tags, boxes, schema) to /Users/luna/Projects/Thesis/second-vision-ai/dataset/backups/review_roboflow_door_detection_zqt59_20260826_183206


In [ ]:
fo.close_app()


NameError: name 'fo' is not defined

# Write back your review edits

**2026-08-26 (DEC-092): back up (the `backup_dir` snapshot cell above) immediately before running this, every time -- do not assume the live dataset here still matches what you last confirmed, even with no kernel restart in between.** A real incident found `accept` tags on predictions silently drop between a confirmed-live count and write-back time (449 -> 208, ~15 minutes, no restart, no rebuild) -- root cause was never found despite directly ruling out a stale `FORCE_REBUILD`, wrong-field mis-tagging, and a duplicate kernel (see docs/DECISIONS.md DEC-092 for exactly what was checked and how). If this cell's printed numbers ever look wrong versus what you just confirmed in the App, don't just re-run it hoping for a different answer -- recover from the most recent backup's `samples.json` directly instead (a standalone script re-deriving write-back's exact logic against that static export, same as DEC-092 did) rather than trusting this cell's live read.

**Run this only after you're done editing in the App above, in the same kernel session** (it reads the live `dataset` object — restarting the kernel loses everything, since this dataset is `persistent=False`).

This does **not** touch your original files in `dataset/processed/<source>/labels/`. It writes to a parallel `labels_reviewed/` folder instead, plus prints a per-file summary of what changed, so you can spot-check before deciding to promote anything. Promoting (copying `labels_reviewed/*.txt` over the real `labels/`) is a separate, deliberate step — not automatic — because this write-back path hasn't been used for a real correction pass yet.

**To accept a missing box the `show_predictions` overlay found** (set `show_predictions = True` above first): tag that specific predicted box `accept` in the App — click it (in the Annotation Canvas or the Labels list), tag from there. This cell copies any `accept`-tagged prediction into `ground_truth` before writing, so it becomes a real label like any other. Predictions are never written to disk themselves, so there's no risk of a redundant ground-truth-plus-prediction pair both surviving — only what's in `ground_truth` after promotion counts.

**To mark an image for removal from the dataset entirely** (wrong content, duplicate, doesn't belong — not a box-editing fix): tag the *sample* (not a specific box) `exclude` instead — tag icon above the sample grid works on a selection; also available per-sample in the modal. This cell reads that tag and writes a per-source `dataset/reports/<source>_excluded.json` that `merge.py` checks on its next run — excluded images get dropped from `dataset/merged/` even though `cap_per_class.py` originally selected them, without ever touching `dataset/processed/`. Untag before running this cell if you change your mind; the exclusion list only reflects whatever's currently tagged, merged with any earlier session's exclusions for this source.

Only valid for a plain processed-source `source_key` (same restriction as `flagged_report_path` above) — `merged`/`final` are derived outputs regenerated by other scripts, not something to hand-edit here.

### Optional: check/remove duplicate `ground_truth` boxes

Added 2026-08-26 (DEC-093), after write-back's prediction-promotion turned out to not be idempotent for a while (fixed in the write-back cell below now, but any duplicates it already created before the fix -- or from any other cause -- won't un-duplicate themselves). Safe to run any time, on any source: exact `(label, bounding_box)` match only, so it can never remove two genuinely distinct boxes that just happen to be close -- only true byte-for-byte copies.

In [16]:
# Re-sync `dataset` with the database BEFORE reading anything off it.
#
# FiftyOne caches Sample objects by id inside this Python process. Once the
# kernel has materialised a sample, `for sample in dataset` hands back that SAME
# cached object -- not the document the App has since written to MongoDB. The
# App is a separate process, so every box you draw, move or delete and every tag
# you click lands in the DB while this kernel keeps serving the stale copy.
#
# Measured directly (2026-08-26), not inferred:
#     kernel materialises a sample   -> 1 box,  tags=[]
#     App writes to the DB           -> 2 boxes, tags=['exclude']
#     `for sample in dataset` sees   -> 1 box,  tags=[]      <-- stale
#     after dataset.reload()         -> 2 boxes, tags=['exclude']
#
# This is why write-back kept "losing" edits while a backup taken moments earlier
# held them: dataset.export() queries MongoDB directly and is always accurate,
# iteration is not. Restoring that backup and writing back from the fresh object
# worked because the restore built uncached Sample objects. It also explains
# DEC-092's "unexplained" tag loss (449 -> 208) -- nothing was ever lost, the
# kernel was reading a snapshot from before those tags existed.
#
# reload() is cheap and always safe: it re-reads documents from the DB and
# discards nothing you have not already saved.
dataset.reload()

dup_count = 0
for sample in dataset:
    seen = set()
    kept = []
    for det in sample.ground_truth.detections:
        key = (det.label, tuple(round(v, 4) for v in det.bounding_box)) if det.bounding_box else (det.label, None)
        if key in seen:
            dup_count += 1
            continue
        seen.add(key)
        kept.append(det)
    if len(kept) != len(sample.ground_truth.detections):
        sample.ground_truth.detections = kept
        sample.save()
        
print(f"Removed {dup_count} duplicate ground_truth detection(s).")

Removed 0 duplicate ground_truth detection(s).


### Optional: retroactively mark already-promoted predictions

Only needed for a source whose predictions were promoted **before** the DEC-093 idempotency fix existed — those are tagged `accept` but never got the `promoted` tag, so write-back would promote them all over again. This finds any `accept`-tagged prediction whose box already exists in `ground_truth` and tags it `promoted` **without adding anything**. Prints 0 on a source reviewed entirely after the fix (i.e. every new source from here on) — harmless to run either way.

In [17]:
# Re-sync `dataset` with the database BEFORE reading anything off it.
#
# FiftyOne caches Sample objects by id inside this Python process. Once the
# kernel has materialised a sample, `for sample in dataset` hands back that SAME
# cached object -- not the document the App has since written to MongoDB. The
# App is a separate process, so every box you draw, move or delete and every tag
# you click lands in the DB while this kernel keeps serving the stale copy.
#
# Measured directly (2026-08-26), not inferred:
#     kernel materialises a sample   -> 1 box,  tags=[]
#     App writes to the DB           -> 2 boxes, tags=['exclude']
#     `for sample in dataset` sees   -> 1 box,  tags=[]      <-- stale
#     after dataset.reload()         -> 2 boxes, tags=['exclude']
#
# This is why write-back kept "losing" edits while a backup taken moments earlier
# held them: dataset.export() queries MongoDB directly and is always accurate,
# iteration is not. Restoring that backup and writing back from the fresh object
# worked because the restore built uncached Sample objects. It also explains
# DEC-092's "unexplained" tag loss (449 -> 208) -- nothing was ever lost, the
# kernel was reading a snapshot from before those tags existed.
#
# reload() is cheap and always safe: it re-reads documents from the DB and
# discards nothing you have not already saved.
dataset.reload()

retro_promoted = 0
for sample in dataset:
    if not sample.has_field("predictions") or sample["predictions"] is None:
        continue
    gt_keys = {
        (d.label, tuple(round(v, 4) for v in d.bounding_box)) if d.bounding_box else (d.label, None)
        for d in sample.ground_truth.detections
    }
    changed = False
    for det in sample["predictions"].detections:
        if "accept" not in (det.tags or []) or "promoted" in (det.tags or []):
            continue
        key = (det.label, tuple(round(v, 4) for v in det.bounding_box)) if det.bounding_box else (det.label, None)
        if key in gt_keys:
            det.tags = [t for t in (det.tags or []) if t != "promoted"] + ["promoted"]
            retro_promoted += 1
            changed = True
    if changed:
        sample.save()

print(f"Retroactively marked {retro_promoted} already-promoted prediction(s) as 'promoted' -- no new boxes added.")


Retroactively marked 0 already-promoted prediction(s) as 'promoted' -- no new boxes added.


In [74]:
# Re-sync `dataset` with the database BEFORE reading anything off it.
#
# FiftyOne caches Sample objects by id inside this Python process. Once the
# kernel has materialised a sample, `for sample in dataset` hands back that SAME
# cached object -- not the document the App has since written to MongoDB. The
# App is a separate process, so every box you draw, move or delete and every tag
# you click lands in the DB while this kernel keeps serving the stale copy.
#
# Measured directly (2026-08-26), not inferred:
#     kernel materialises a sample   -> 1 box,  tags=[]
#     App writes to the DB           -> 2 boxes, tags=['exclude']
#     `for sample in dataset` sees   -> 1 box,  tags=[]      <-- stale
#     after dataset.reload()         -> 2 boxes, tags=['exclude']
#
# This is why write-back kept "losing" edits while a backup taken moments earlier
# held them: dataset.export() queries MongoDB directly and is always accurate,
# iteration is not. Restoring that backup and writing back from the fresh object
# worked because the restore built uncached Sample objects. It also explains
# DEC-092's "unexplained" tag loss (449 -> 208) -- nothing was ever lost, the
# kernel was reading a snapshot from before those tags existed.
#
# reload() is cheap and always safe: it re-reads documents from the DB and
# discards nothing you have not already saved.
dataset.reload()

if source_key == "merged" or source_key.startswith("final/"):
    raise ValueError(
        "Write-back is only supported against a plain processed-source source_key -- "
        "merged/final are derived outputs regenerated by other scripts, not hand-edited here."
    )

orig_labels_dir = processed_dir(source_key) / "labels"
out_labels_dir = processed_dir(source_key) / "labels_reviewed"
# No longer clears labels_reviewed/ first (used to: shutil.rmtree then rebuild fresh).
# labels_reviewed/ is now a CUMULATIVE record across sessions with potentially different
# scopes -- e.g. a 5,500-per-class review now, then a wider 10,000 review later that
# should skip what's already here (skip_previously_reviewed, build cell above). Clearing
# it on every run would destroy that record for any image outside THIS session's scope
# the moment the next write-back ran. Only ever writes/overwrites files for samples
# currently in `dataset` below -- anything already in labels_reviewed/ for an
# out-of-scope image is left untouched, not orphaned-and-deleted.
ensure_dir(out_labels_dir)

# Promote accepted predictions into ground_truth first, so the write logic
# below (which only ever reads ground_truth) picks them up automatically.
# Only relevant when show_predictions was True above -- a no-op otherwise,
# since samples won't have a populated predictions field to promote from.
# Appends rather than replaces, so existing ground_truth boxes are untouched
# either way. Predictions themselves are never written to disk anywhere --
# only what ends up in ground_truth (including anything promoted here)
# becomes a real label, so there's no risk of a redundant ground-truth +
# prediction pair both surviving into labels_reviewed/.
promoted = 0
for sample in dataset:
    if not sample.has_field("predictions") or sample["predictions"] is None:
        continue
    # Only "accept" (not "promoted") -- a prediction already promoted in an earlier
    # write-back run gets its tag flipped to "promoted" below, specifically so this
    # filter naturally excludes it on the next run. Bug fixed 2026-08-26 (DEC-093):
    # this used to re-check only "accept" with nothing ever clearing it, so every
    # write-back re-promoted every previously-promoted prediction again -- a second
    # run duplicated every box from the first, a third run tripled them, etc.
    accepted = [
        det for det in sample["predictions"].detections
        if "accept" in (det.tags or []) and "promoted" not in (det.tags or [])
    ]
    if not accepted:
        continue
    sample["ground_truth"].detections.extend(
        fo.Detection(label=det.label, bounding_box=det.bounding_box) for det in accepted
    )
    # Mark these predictions promoted so a future write-back run's filter above
    # skips them -- "accept" is left in place as a historical record of intent,
    # "promoted" is the new idempotency guard.
    for det in accepted:
        det.tags = [t for t in (det.tags or []) if t != "promoted"] + ["promoted"]
    sample.save()
    promoted += len(accepted)
if promoted:
    print(f"Promoted {promoted} accepted prediction(s) into ground_truth (tagged 'accept' in the App).\n")

# Exclusion tracking: samples tagged "exclude" in the App get merged into this
# source's own dataset/reports/<source>_excluded.json, which merge.py reads to
# drop them from the next dataset/merged/ rebuild -- without touching
# dataset/processed/ at all. Merged with any existing file, not overwritten --
# a full-pool review can span many sittings, and an earlier sitting's
# exclusions must survive a later one's write-back.
excluded_json_path = reports_dir() / f"{source_key.removeprefix('roboflow_')}_excluded.json"
existing_excluded: set[str] = set()
if excluded_json_path.is_file():
    existing_excluded = set(json.loads(excluded_json_path.read_text(encoding="utf-8"))["excluded_filenames"])

added = removed = modified = unchanged = 0
newly_excluded: set[str] = set()
# Detections with missing/malformed geometry (e.g. a click-without-drag in the
# Annotate tab leaving bounding_box == []) get skipped rather than crashing the
# whole write-back -- finding one bad box in a pool of ~1000 images by hand isn't
# reasonable to ask of the student mid-review. Tracked here so it's surfaced as a
# specific, fixable warning (go re-draw that box in the App) instead of silently
# dropped or silently blocking every other file's progress.
skipped_malformed: list[tuple[str, str, str]] = []
for sample in dataset:
    label_filename = sample["source_label_filename"]
    if "exclude" in sample.tags:
        newly_excluded.add(label_filename)

    orig_path = orig_labels_dir / label_filename
    orig_lines = [
        ln.strip() for ln in (orig_path.read_text(encoding="utf-8").splitlines() if orig_path.is_file() else [])
        if ln.strip()
    ]

    new_lines = []
    for det in sample.ground_truth.detections:
        if det.label not in CANONICAL_NAMES:
            raise ValueError(
                f"{label_filename}: detection has label {det.label!r}, not one of the 16 canonical "
                f"classes -- check for a typo introduced via the App's class dropdown before re-running."
            )
        if not det.bounding_box or len(det.bounding_box) != 4:
            skipped_malformed.append((label_filename, det.id, det.label))
            continue
        class_id = CANONICAL_NAMES.index(det.label)
        # Inverse of the load cell's transform: FiftyOne's top-left [x, y, w, h] -> YOLO center-based.
        x, y, w, h = det.bounding_box
        cx, cy = x + w / 2, y + h / 2
        new_lines.append(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

    (out_labels_dir / label_filename).write_text(
        "\n".join(new_lines) + ("\n" if new_lines else ""), encoding="utf-8"
    )

    if len(new_lines) > len(orig_lines):
        added += 1
    elif len(new_lines) < len(orig_lines):
        removed += 1
    elif set(new_lines) != set(orig_lines):
        modified += 1
    else:
        unchanged += 1

all_excluded = sorted(existing_excluded | newly_excluded)
excluded_json_path.write_text(
    json.dumps({"source": source_key, "excluded_filenames": all_excluded}, indent=2),
    encoding="utf-8",
)

print(f"Wrote {len(dataset)} label files to {out_labels_dir}")
print(
    f"vs. originals in {orig_labels_dir}: "
    f"{added} files gained box(es), {removed} files lost box(es), "
    f"{modified} files changed geometry/class only (same count), {unchanged} untouched"
)
if skipped_malformed:
    print(
        f"\nWARNING: {len(skipped_malformed)} detection(s) skipped -- missing/malformed "
        f"bounding_box (probably a click-without-drag in the Annotate tab). Not written to "
        f"labels_reviewed/ at all, so this is a real box you'll want to go re-draw:"
    )
    for fn, det_id, label in skipped_malformed:
        print(f"    {fn}  detection id={det_id}  label={label!r}")
if newly_excluded:
    print(
        f"\n{len(newly_excluded)} newly tagged 'exclude' this run "
        f"({len(all_excluded)} total for {source_key}) -> {excluded_json_path}\n"
        "These still got a labels_reviewed/ file above (in case you untag and change your mind "
        "before promoting), but the next merge.py run will drop them from dataset/merged/ "
        "regardless of what you promote -- they won't reach the trained model unless untagged first."
    )
print(
    "\nNothing in dataset/processed/<source>/labels/ has been touched yet. Spot-check "
    "labels_reviewed/ (e.g. re-point source_key's flagged_report_path-free run at it, or diff "
    "a few files by hand), then explicitly copy the ones you're confident in over the real "
    "labels/ folder when ready to promote -- that promotion step is intentionally manual."
)

Promoted 77 accepted prediction(s) into ground_truth (tagged 'accept' in the App).

Wrote 607 label files to /Users/luna/Projects/Thesis/second-vision-ai/dataset/processed/roboflow_door_detection_zqt59/labels_reviewed
vs. originals in /Users/luna/Projects/Thesis/second-vision-ai/dataset/processed/roboflow_door_detection_zqt59/labels: 63 files gained box(es), 0 files lost box(es), 0 files changed geometry/class only (same count), 544 untouched

1 newly tagged 'exclude' this run (1 total for roboflow_door_detection_zqt59) -> /Users/luna/Projects/Thesis/second-vision-ai/dataset/reports/door_detection_zqt59_excluded.json
These still got a labels_reviewed/ file above (in case you untag and change your mind before promoting), but the next merge.py run will drop them from dataset/merged/ regardless of what you promote -- they won't reach the trained model unless untagged first.

Nothing in dataset/processed/<source>/labels/ has been touched yet. Spot-check labels_reviewed/ (e.g. re-point sour

In [19]:
# Visual sanity check: render labels_reviewed/ as actual bounding boxes, to confirm
# the write-back's coordinate round-trip (FiftyOne top-left [x,y,w,h] -> YOLO
# center-based -- the inverse of what the write-back cell just did) produced boxes
# that actually look right, not just numbers that happen to parse. Read-only: builds
# a separate, throwaway dataset straight from disk and does not touch `dataset` or
# `session` above, so it's safe to run any time after a write-back without disturbing
# an in-progress review. Reuses the exact same YOLO-parsing logic as the main build
# cell (proven correct there), just pointed at labels_reviewed/ instead of labels/.
#
# Excluded images are shown TAGGED, not silently mixed in. Write-back deliberately
# still writes a labels_reviewed/ file for an image tagged 'exclude' -- the exclusion
# is recorded in dataset/reports/<source>_excluded.json, and merge.py subtracts those
# (source, stem) pairs from the pool (scripts/build/merge.py:125-172). This cell used
# to load every file on disk with no reference to that JSON, so every excluded image
# reappeared here looking exactly like a failed exclusion. It is not; it is the
# division of labour working as designed.
VERIFY_HIDE_EXCLUDED = True

verify_dataset_name = f"verify_{source_key.replace('/', '_')}_reviewed"
if verify_dataset_name in fo.list_datasets():
    fo.delete_dataset(verify_dataset_name)
verify_dataset = fo.Dataset(verify_dataset_name, persistent=False)

verify_labels_dir = processed_dir(source_key) / "labels_reviewed"
if not verify_labels_dir.is_dir():
    raise FileNotFoundError(f"{verify_labels_dir} not found -- run the write-back cell above first.")

verify_excluded_path = reports_dir() / f"{source_key.removeprefix('roboflow_')}_excluded.json"
verify_excluded_stems: set[str] = set()
if verify_excluded_path.is_file():
    verify_excluded_stems = {
        Path(name).stem
        for name in json.loads(verify_excluded_path.read_text(encoding="utf-8"))["excluded_filenames"]
    }

verify_samples = []
skipped_no_reviewed_file = 0
for image_path in sorted((processed_dir(source_key) / "images").iterdir()):
    label_path = verify_labels_dir / f"{image_path.stem}.txt"
    if not label_path.is_file():
        skipped_no_reviewed_file += 1
        continue  # this image wasn't part of the write-back run (e.g. restrict_to_merged skipped it)

    sample = fo.Sample(filepath=str(image_path))
    if image_path.stem in verify_excluded_stems:
        sample.tags = ["excluded"]
    detections = []
    for line in label_path.read_text(encoding="utf-8").splitlines():
        parts = line.strip().split()
        if not parts:
            continue
        class_id = int(parts[0])
        cx, cy, w, h = (float(v) for v in parts[1:5])
        x, y = cx - w / 2, cy - h / 2
        detections.append(fo.Detection(label=CANONICAL_NAMES[class_id], bounding_box=[x, y, w, h]))
    sample["ground_truth"] = fo.Detections(detections=detections)
    verify_samples.append(sample)

verify_dataset.add_samples(verify_samples)
verify_dataset.classes = {"ground_truth": CANONICAL_NAMES}
verify_dataset.save()

n_excluded = len(verify_dataset.match_tags("excluded"))
print(f"{len(verify_dataset)} images loaded from {verify_labels_dir} (read-only visual check)")
if skipped_no_reviewed_file:
    print(f"({skipped_no_reviewed_file} images skipped -- no labels_reviewed/ file, not part of this write-back run)")
if verify_excluded_stems:
    print(
        f"{n_excluded} of them are tagged 'excluded' per {verify_excluded_path.name} "
        f"({len(verify_excluded_stems)} listed there) -- merge.py drops these from the pool."
    )
    if n_excluded != len(verify_excluded_stems):
        print(
            f"  NOTE: {len(verify_excluded_stems) - n_excluded} listed exclusion(s) matched no image here, "
            f"which is normal if this write-back covered a narrower slice than an earlier one."
        )
else:
    print(f"No {verify_excluded_path.name} on disk -- nothing was excluded for this source.")

# Separate port from the main review App (default 5151) so both can stay open at
# once without one replacing the other in your browser.
verify_session = fo.launch_app(verify_dataset, auto=False, port=5152)
if VERIFY_HIDE_EXCLUDED and n_excluded:
    verify_session.view = verify_dataset.match_tags("excluded", bool=False)
    print(f"Showing the {len(verify_session.view)} kept images; set VERIFY_HIDE_EXCLUDED = False to see all.")
print("Verify App: http://localhost:5152/")


 100% |█████████████████| 665/665 [247.7ms elapsed, 0s remaining, 2.7K samples/s]      
665 images loaded from /Users/luna/Projects/Thesis/second-vision-ai/dataset/processed/roboflow_pothole_voxrl/labels_reviewed (read-only visual check)
Session launched. Run `session.show()` to open the App in a cell output.
Verify App: http://localhost:5152/


# Mistakenness review (Stage 5.5, top-N) — a different mode from everything above

Everything above is single-source (`source_key`). This section is deliberately different: `dataset/reports/mistakenness_report.json` ranks **22,846 images across all 7 COCO-eligible classes and every source that contributes to them**, by how much a stock COCO-pretrained `yolov8n.pt` disagrees with your ground truth. Reviewing all 22,846 isn't the intent — reviewing a bounded top slice, ranked worst-first, is.

**What "eligible" means here, concretely:** only Person, Vehicle, Motorcycle, Bicycle, Animals, Chairs, Tables have a COCO analog (see `scripts/curate/run_mistakenness.py`'s module docstring for the exact crosswalk and why). The other 9 classes get zero signal from this — this section can't help you review Doors, Elevator, Stairs, etc.

**Predictions are computed fresh here, not read from the report** (the report only stored counts, not box coordinates) — same model, same crosswalk, same logic as `run_mistakenness.py`, imported directly so this can't silently drift from what actually produced the ranking. For ~1,000 images this is a couple of minutes on this machine, not a background-task situation.

Each sample shows **both** `ground_truth` (eligible classes only — that's what was actually scored) and `predictions` (the proxy model's boxes, with confidence) side by side, plus the real `mistakenness` score as a field you can sort/filter by in the App sidebar. Seeing *why* an image was flagged (what the model saw vs. what you labeled) is the point — that's different from the box-audit review above, which is about box geometry, not a disagreement between two label sets.

**Note:** this section's write-back (below) only ever reads `ground_truth`, never `predictions` — there's no accept-tag promotion here like the box-audit section above, so tagging a prediction `accept` in this App does nothing. Edit `ground_truth` directly instead.

In [ ]:
# How many of the top-ranked images to pull in for this review pass.
MISTAKENNESS_TOP_N = 1000

mistakenness_report = json.loads(
    (REPO_ROOT / "dataset/reports/mistakenness_report.json").read_text(encoding="utf-8")
)
top_ranked = mistakenness_report["ranked"][:MISTAKENNESS_TOP_N]
print(
    f"{len(top_ranked)} images selected "
    f"(top {MISTAKENNESS_TOP_N} of {mistakenness_report['unique_images_scored']} scored)"
)

In [ ]:
# Build: resolve each top-ranked image, load its eligible-class ground truth, run fresh
# yolov8n inference for predictions. Mirrors run_mistakenness.py's own inference loop
# exactly (same model, same crosswalk, same device selection) -- imported rather than
# reimplemented for everything that isn't image-loading itself.
from ultralytics import YOLO
import torch

model = YOLO("yolov8n.pt")
coco_to_canonical = {c: CANONICAL_KEY_TO_NAME[key] for key, cs in COCO_CROSSWALK.items() for c in cs}
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

sources_needed = {r["source"] for r in top_ranked}
stem_indexes = {source: build_stem_index(source) for source in sources_needed}

records = []
for r in top_ranked:
    img_path = stem_indexes[r["source"]].get(r["filename"])
    if img_path is None:
        continue  # shouldn't happen against a consistent processed/ pool, but don't hard-fail a review session over it
    records.append((r["source"], r["filename"], img_path, r["mistakenness"]))

paths = [str(p) for _, _, p, _ in records]
BATCH = 16
predictions_by_path: dict[str, list[dict]] = {}
print(f"Running yolov8n inference on {len(paths)} images (device={device})...")
for i in range(0, len(paths), BATCH):
    batch = paths[i:i + BATCH]
    results = model.predict(batch, device=device, verbose=False)
    for path, result in zip(batch, results):
        dets = []
        names = result.names
        for box in result.boxes:
            coco_name = names[int(box.cls.item())]
            if coco_name not in coco_to_canonical:
                continue
            x1, y1, x2, y2 = box.xyxyn[0].tolist()
            dets.append({
                "label": coco_to_canonical[coco_name],
                "bounding_box": [x1, y1, x2 - x1, y2 - y1],
                "confidence": float(box.conf.item()),
            })
        predictions_by_path[path] = dets

print("Inference complete. Building FiftyOne dataset...")
mistakenness_dataset_name = "review_mistakenness_top_n"
if mistakenness_dataset_name in fo.list_datasets():
    fo.delete_dataset(mistakenness_dataset_name)
mistakenness_dataset = fo.Dataset(mistakenness_dataset_name, persistent=False)

samples = []
for source, filename, img_path, score in records:
    gt_boxes = load_eligible_ground_truth(source, filename, CANONICAL_NAMES)
    pred_boxes = predictions_by_path.get(str(img_path), [])
    sample = fo.Sample(filepath=str(img_path))
    # Stashed for the write-back cell -- which source/file this sample's edits belong to.
    sample["source"] = source
    sample["source_filename"] = filename
    sample["mistakenness"] = score
    sample["ground_truth"] = fo.Detections(detections=[fo.Detection(**b) for b in gt_boxes])
    sample["predictions"] = fo.Detections(detections=[fo.Detection(**b) for b in pred_boxes])
    samples.append(sample)

mistakenness_dataset.add_samples(samples)
print(f"{len(mistakenness_dataset)} images loaded, ready to review (sort by the mistakenness field in the App to see worst-first)")

In [ ]:
# Launch the App for the mistakenness review. Click the mistakenness field in the
# left sidebar to sort worst-first if it doesn't default to it.
mistakenness_session = fo.launch_app(mistakenness_dataset, auto=False)

# Write back mistakenness review edits

Same rule as the box-audit write-back above: run this after reviewing in the App, same kernel session, before restarting the kernel.

**One real difference from the box-audit write-back, worth understanding before you trust it:** `ground_truth` here only ever held the 7 COCO-eligible classes' boxes (that's what mistakenness was computed against) — an image can have other canonical-class boxes (e.g. a Pole box on a Person-eligible image) that were never loaded into this dataset at all. So this cell doesn't just dump `ground_truth` back out — it reads each original label file fresh, keeps every non-eligible-class line exactly as it was, and only replaces the eligible-class lines with whatever's currently in `ground_truth` (i.e., your edits). Verified before being handed to you: simulated deleting an eligible box on a real mixed-class file and confirmed the non-eligible line survived byte-for-byte and the original file on disk was untouched.

Writes to each source's own `dataset/processed/<source>/labels_reviewed/` — the same staging convention as above, shared across both review modes. Unlike the box-audit write-back, this one does **not** clear the whole `labels_reviewed/` folder first, since a 1,000-image mistakenness pass may span multiple sittings across many sources and shouldn't wipe out another review's already-promoted files each time it re-runs.

In [9]:
changed_by_source: dict[str, int] = {}
diff_added = diff_removed = diff_modified = diff_unchanged = 0

for sample in mistakenness_dataset:
    source = sample["source"]
    filename = sample["source_filename"]
    orig_label_path = processed_dir(source) / "labels" / f"{filename}.txt"
    out_labels_dir = processed_dir(source) / "labels_reviewed"
    ensure_dir(out_labels_dir)

    full_orig_lines = [
        ln.strip() for ln in (orig_label_path.read_text(encoding="utf-8").splitlines() if orig_label_path.is_file() else [])
        if ln.strip()
    ]
    # Keep every non-eligible-class line exactly as it was -- this review never loaded them,
    # so they can't have been edited, and must not be dropped.
    non_eligible_lines = [ln for ln in full_orig_lines if int(ln.split()[0]) not in ELIGIBLE_CANONICAL_IDS]

    new_eligible_lines = []
    for det in sample.ground_truth.detections:
        if det.label not in CANONICAL_NAMES:
            raise ValueError(
                f"{source}/{filename}: detection has label {det.label!r}, not one of the 16 canonical "
                f"classes -- check for a typo introduced via the App's class dropdown before re-running."
            )
        class_id = CANONICAL_NAMES.index(det.label)
        x, y, w, h = det.bounding_box
        cx, cy = x + w / 2, y + h / 2
        new_eligible_lines.append(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

    merged_lines = non_eligible_lines + new_eligible_lines
    (out_labels_dir / f"{filename}.txt").write_text(
        "\n".join(merged_lines) + ("\n" if merged_lines else ""), encoding="utf-8"
    )
    changed_by_source[source] = changed_by_source.get(source, 0) + 1

    if len(merged_lines) > len(full_orig_lines):
        diff_added += 1
    elif len(merged_lines) < len(full_orig_lines):
        diff_removed += 1
    elif set(merged_lines) != set(full_orig_lines):
        diff_modified += 1
    else:
        diff_unchanged += 1

print(f"Wrote {sum(changed_by_source.values())} label files across {len(changed_by_source)} sources:")
for source, count in sorted(changed_by_source.items()):
    print(f"  {source}: {count} files -> {processed_dir(source) / 'labels_reviewed'}")
print(
    f"\nvs. originals: {diff_added} files gained box(es), {diff_removed} files lost box(es), "
    f"{diff_modified} files changed geometry/class only (same count), {diff_unchanged} untouched"
)
print(
    "\nNothing in any dataset/processed/<source>/labels/ has been touched. Spot-check "
    "labels_reviewed/ per source, then explicitly copy the files you're confident in over "
    "the real labels/ folder when ready to promote -- same manual promotion step as above."
)

NameError: name 'mistakenness_dataset' is not defined

**Review checklist for dataset review #1— flag-rate percentages (DEC-076), image counts from `dataset/merged/`.** Listed by measured `box_audit.py --pool merged` flag rate, high to low — a pacing guide for your own attention, not a priority/skip list (every source still gets a full pass, none skipped). "Reviewed" means through write-back completing cleanly — promoting `labels_reviewed/` over the real `labels/` is a separate, later step done across sources together, not tracked per-row here. Toggle the checkboxes yourself as you finish a source (or ask Claude to).

**Operating policy update, 2026-08-21 (DEC-087):** student's standing rule — any source still `[x]` here (i.e. never reached `[/]`, a completed write-back review) gets deactivated rather than reviewed, once a suitable replacement/alternative source exists for its class. Applied so far: `me5_u6rvg` and `augmented_tricycle` (both benched). This does NOT mean every `[x]` row below is confirmed bad — several were marked `[x]` purely because they hadn't gone through the full in-App pass yet, not because of a specific found defect (`stairs_i2yia`/`escalator_stairs`/`elevator_status_s4lrk` are the exception — those have real, independently-verified defects, see DEC-082). Check `config/datasets.yaml`'s `audit_status` for any given source's actual current state — this checklist tracks review progress, not the authoritative active/inactive status.

- [o] `door_detection_zqt59` — 15.88%, 3,460 images *(review started once, but tags were lost to a kernel restart before write-back completed — needs a fresh pass)*
- [/] `cv_project_hovyc` — 13.63%, 1,040 images *(reviewed, write-back complete — 71 excluded, 277 promoted. Promoted over the real `labels/` 2026-08-21, DEC-087 — this source's cleaned version is now what merge.py will read.)*
- [x] `stairs_i2yia` — 12.32%, 1,195 images *(boxes found systematically mispositioned and wrong-shaped on every image checked, not just DEC-031's shape-only defect — dropped entirely, `audit_status: failed`, DEC-082)*
- [x] `elevator_status_s4lrk` — 12.20%, 2,723 images *(DEC-031's box-shape concern, independently re-confirmed and worse via direct visual sampling 2026-08-20 — corrupted black regions baked into the source images themselves. Dropped entirely, `audit_status: failed`, DEC-082.)*
- [x] `pothole_vhmow` — 12.11%, 871 images *(benched 2026-08-20, DEC-083 — superseded by pothole_voxrl, not confirmed defective)*
- [x] `escalator_stairs` — 10.79%, 7,560 images *(DEC-031, same defect as elevator_status_s4lrk — direct visual check 2026-08-20 confirmed low-res/grainy/near-duplicate content on its Stairs slice. Dropped entirely, `audit_status: failed`, DEC-082.)*
- [x] `pedestrian_and_animal_crossing` — 10.23%, 2,158 images *(benched 2026-08-20, DEC-083 — superseded by wtf_dwvgm + revised_pedestrian_obstacle, not confirmed defective)*
- [x] `elevator_awvus` — 9.31%, 1,777 images *(this row's original note cited DEC-031's box-shape concern — since independently re-checked via direct visual sampling, 2026-08-20: 3/3 sampled images showed correctly-placed, tight boxes on real elevator doors. DEC-082 kept this active as the clean, primary Elevator source — the DEC-031 concern does not hold for this source specifically. Still hasn't gone through a full write-back pass here, but per DEC-087 this is NOT being deactivated despite the `[x]` mark — the direct visual evidence outweighs "unreviewed.")*
- [x] `utility_poles_44tzx` — 8.44%, 2,795 images *(benched 2026-08-20, DEC-083 — Pole switched to Open Images; its real Potholes contribution was already zero, not confirmed defective)*
- [x] `pole_detection_z76mb` — 7.97%, 1,705 images *(benched 2026-08-20, DEC-083 — Pole switched to Open Images, not confirmed defective)*
- [x] `augmented_tricycle` — 6.16%, 3,021 images *(benched 2026-08-21, DEC-087 — never reviewed, student's call to drop per the operating-policy update above rather than review it)*
- [x] `me5_u6rvg` — 4.49%, 4,371 images *(benched 2026-08-21, DEC-087 — never reviewed, same call as augmented_tricycle; superseded by dlsu_d_vehicle_type_detection across Vehicle/Motorcycle/Tricycle)*
- [/] `trashcan_detection_pihfn` — 3.54%, 559 images *(reviewed, write-back complete. Promoted over the real `labels/` 2026-08-21, DEC-087 — this source's cleaned version is now what merge.py will read.)*

**Not yet on this list** (added after DEC-076's ranking, no flag-rate/review status yet): `stair_gaptw`, `elevator_status_0iq4p`, `pothole_voxrl`, `wtf_dwvgm`, `revised_pedestrian_obstacle`, `dlsu_d_vehicle_type_detection`, `roitrikee` — all 7 added 2026-08-20/21 (DEC-082/083). Will need their own `box_audit.py --pool merged` pass and risk ranking added here once the cap/merge cascade actually re-runs with them included.

**Review checklist for dataset review #2 — post-dedup, 5,500-cap slice (DEC-091).** Extended this pass to every active source, not just Roboflow (student's explicit request 2026-08-24) — Roboflow sources listed first (this project's original review-priority group), non-Roboflow sources at the bottom (lower priority per the student's own framing). Ranked within each group by measured `box_audit.py --pool merged` flag rate, high to low — same pacing-guide convention as review #1's checklist (scroll down), not a priority/skip list.

**Scope note on the two numbers per row:** flag-rate % is from a fresh `box_audit.py --pool merged` run (2026-08-24) against whatever's *currently* on disk at `dataset/merged/` — at the time this was written, that's still the 10,000-cap pool (67,109 images), since `merge.py --cap-report cap_report_hardcap5500.json` hasn't been run yet to physically rebuild `dataset/merged/` at the 5,500-cap level. Image counts, by contrast, ARE the real 5,500-cap-slice numbers — each source's unique image count in `dataset/reports/cap_report_hardcap5500.json`'s selection (deduped across classes), i.e. what this review pass is actually scoped to review once the merge catches up. Re-run `box_audit.py --pool merged` again after that merge for exact-to-scope flag rates if it matters; the ranking itself is unlikely to reshuffle much.

**Checkbox legend:** `[ ]` not started this pass · `[o]` started but incomplete · `[/]` reviewed + write-back complete · `[x]` inactive/benched (shown below the list for reference only, not part of this pass's queue).

**Promotion status is tracked separately from review status** (added 2026-08-26 — it turned out not to be uniform, contrary to an easy assumption that every reviewed source sits in the same phase). *Reviewed* means write-back wrote `labels_reviewed/` cleanly. *Promoted* means `labels_reviewed/` was copied over the real `labels/` — and `labels/` is the **only** thing `cap_per_class.py` and `merge.py` actually read (verified directly 2026-08-26: nothing anywhere under `scripts/` references `labels_reviewed/` at all, so an unpromoted review has zero effect on the training pool). **Promoted:** `cv_project_hovyc`, `trashcan_detection_pihfn` (both 2026-08-21, DEC-087). **Reviewed but NOT promoted:** `roboflow_pothole_voxrl`. Promoting changes a source's class content, which changes `cap_per_class.py`'s selection — so the cap → merge cascade must be re-run after any promotion, before `split.py`.

**Roboflow sources (12):**

- [/] `roboflow_pothole_voxrl` — 16.79%, 665 images *(reviewed 2026-08-26, write-back complete: 2,262 boxes in `labels_reviewed/`, verified free of exact duplicates. **Not yet promoted** — see the promotion note above. The review turned this from a single-class Potholes source into a 10-class one (Potholes 1,739 unchanged, plus Vehicle 375, Person 84, Pole 20, Motorcycle 18, Tricycle 14, Bicycle 6, Pedestrian Lane 3, Stairs 2, Doors 1) — same "no longer single-class after review" pattern DEC-087 found for hovyc/trashcan. Known minor loose end: 3 double-prediction pairs left in (`img-286`, `img-521`, `img-596`, all Vehicle, IoU 0.81–0.95) — two accepted predictions of the same car, not the DEC-093 duplication bug. This source's review is also what surfaced DEC-092 and DEC-093 — read both before trusting any write-back's printed numbers at face value.)*
- [/] `roboflow_door_detection_zqt59` — 16.10%, 4,240 images *(review #1: started once, but tags were lost to a kernel restart before write-back completed — needs a genuinely fresh pass)*
- [ ] `roboflow_revised_pedestrian_obstacle` — 13.36%, 3,960 images
- [o] `roboflow_elevator_status_0iq4p` — 12.97%, 1,414 images *(added DEC-082, never reviewed here yet)*
- [/] `roboflow_cv_project_hovyc` — 11.84%, 1,271 images *(reviewed, write-back complete, promoted over `labels/` 2026-08-21 DEC-087 — this pass only needs a light recheck of whichever of its images actually land in the final 5,500/4,500 selection, not a full re-review)*
- [ ] `roboflow_dlsu_d_vehicle_type_detection` — 10.88%, 10,613 images *(added DEC-083, never reviewed here yet — largest Roboflow source in this pass, pace accordingly)*
- [o] `roboflow_wtf_dwvgm` — 9.67%, 1,326 images *(added DEC-083, never reviewed here yet)*
- [o] `roboflow_elevator_awvus` — 9.31%, 1,777 images *(never went through a full write-back pass, but DEC-082's direct visual spot-check — 3/3 sampled images — found correctly-placed, tight boxes on real elevator doors; kept active on that evidence despite the `[ ]` state)*
- [o] `roboflow_crosswalk_detector_lz3hc` — 9.29%, 202 images *(newest source, added DEC-090 to close Pedestrian Lane's floor shortfall — has a known rotated-image/misaligned-box defect on some images, portrait content stored as a landscape pixel grid with no EXIF. Finishing this source's exclude-tagging is this pass's specific unfinished business, deferred from DEC-090.)*
- [o] `roboflow_stair_gaptw` — 8.61%, 1,564 images *(added DEC-082, never reviewed here yet)*
- [ ] `roboflow_roitrikee` — 5.81%, 665 images *(added DEC-083, never reviewed here yet)*
- [/] `roboflow_trashcan_detection_pihfn` — 5.67%, 559 images *(reviewed, write-back complete, promoted over `labels/` 2026-08-21 DEC-087 — same light-recheck note as cv_project_hovyc above)*

**Non-Roboflow sources (5) — lower priority, never been on a checklist here before:**

- [ ] `dataset_ninja_pothole_detection` — 16.73%, 665 images *(activated as the Potholes floor fallback 2026-08-21, DEC-086 — never reviewed in this notebook)*
- [ ] `dataset_ninja_road_damage_detector` — 16.48%, 1,331 images
- [ ] `open_images` — 15.59%, 18,847 images *(largest single source in this entire pass by a wide margin — pace accordingly, or consider whether a full box-by-box pass is worth it at this volume vs. flagged-only mode)*
- [ ] `exdark` — 11.61%, 5,664 images
- [ ] `crowdhuman` — 5.15%, 458 images

**Inactive/benched, not in this pass's pool** (`audit_status: failed`/`benched` — see `config/datasets.yaml`, review #1's checklist above has the individual history for each): `stairs_i2yia`, `elevator_status_s4lrk`, `escalator_stairs`, `pothole_vhmow`, `pedestrian_and_animal_crossing`, `utility_poles_44tzx`, `pole_detection_z76mb`, `augmented_tricycle`, `me5_u6rvg`.
